In [10]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-29")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

In [11]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

**Problem 1** | Amazon-style

Customers who ordered the same product 3+ times

Amazon-style loyalty question: find customers who have ordered the exact same product_id three or more times. Return customer_id, product_id, times_ordered.

In [12]:
orders_df.groupBy('customer_id','product_id').\
agg(F.countDistinct('order_id').alias('times_orders')).\
filter(F.col('times_orders')>=3).show()

+-----------+----------+------------+
|customer_id|product_id|times_orders|
+-----------+----------+------------+
+-----------+----------+------------+



**Problem 2** | Netflix-style

"Binge" detection — orders within 24 hours of each other

Netflix-style engagement question, adapted: find customers who placed 2 or more orders on the exact same order_date — a proxy for a "binge purchasing" session. Return customer_id, order_date, orders_that_day.

In [13]:
orders_df.groupBy("customer_id", "order_date").agg(
    F.count("order_id").alias("orders_that_day")
).filter(F.col("orders_that_day") >= 2) \
 .orderBy(F.col("orders_that_day").desc()) \
 .show()

+-----------+----------+---------------+
|customer_id|order_date|orders_that_day|
+-----------+----------+---------------+
+-----------+----------+---------------+



**Problem 3** | Uber-style

Supply-demand mismatch — region order volume vs available stock

Uber-style supply/demand question, adapted: for each region, compare total quantity ordered against the stock_quantity available (sum stock across all products). Flag any region where demand (quantity ordered) exceeds 50% of total available stock as "high_demand".

In [14]:
total_stock = products_df.agg(F.sum("stock_quantity")).collect()[0][0]

regional_demand = orders_df.groupBy("region").agg(
    F.sum("quantity").alias("total_demand")
)

regional_demand.withColumn("total_supply", F.lit(total_stock)) \
    .withColumn(
        "demand_status",
        F.when(F.col("total_demand") > 0.5 * F.col("total_supply"), "high_demand").otherwise("normal")
    ).orderBy(F.col("total_demand").desc()) \
    .show()

+-------+------------+------------+-------------+
| region|total_demand|total_supply|demand_status|
+-------+------------+------------+-------------+
|   West|          70|        5645|       normal|
|   East|          68|        5645|       normal|
|Midwest|          50|        5645|       normal|
|  South|          44|        5645|       normal|
+-------+------------+------------+-------------+



**Problem 4** | Meta-style

Funnel analysis — signup to first purchase

Meta-style funnel question: for each customer, calculate days between signup_date and their first order's order_date. Bucket customers into: "Same Day" (0 days), "Within Week" (1-7), "Within Month" (8-30), "Slow" (31+), "Never Purchased" (no orders). Show the count in each bucket.

In [15]:
first_order = orders_df.groupBy("customer_id").agg(
    F.min("order_date").alias("first_order_date")
)

funnel = customers_df.join(first_order, on="customer_id", how="left") \
    .withColumn("days_to_purchase", F.datediff(F.col("first_order_date"), F.col("signup_date"))) \
    .withColumn(
        "funnel_bucket",
        F.when(F.col("first_order_date").isNull(), "Never Purchased")
         .when(F.col("days_to_purchase") == 0, "Same Day")
         .when(F.col("days_to_purchase") <= 7, "Within Week")
         .when(F.col("days_to_purchase") <= 30, "Within Month")
         .otherwise("Slow")
    )

funnel.groupBy("funnel_bucket").agg(F.count("customer_id").alias("customer_count")) \
    .orderBy(F.col("customer_count").desc()) \
    .show()

+-------------+--------------+
|funnel_bucket|customer_count|
+-------------+--------------+
|         Slow|            25|
+-------------+--------------+



**Problem 5** | Spotify-style

"Discovery" metric — first-time category purchases by month

Spotify-style discovery question, adapted: for each customer, identify the first month they ever purchased from each category (their "discovery month" for that category). Count how many "discoveries" happened in each calendar month across all customers.

In [16]:
orders_with_category = orders_df.join(
    products_df.select("product_id", "category"), on="product_id", how="inner"
)

discoveries = orders_with_category.groupBy("customer_id", "category").agg(
    F.min("order_date").alias("discovery_date")
).withColumn("discovery_month", F.date_format("discovery_date", "yyyy-MM"))

discoveries.groupBy("discovery_month").agg(
    F.count("*").alias("num_discoveries")
).orderBy("discovery_month").show(20)

+---------------+---------------+
|discovery_month|num_discoveries|
+---------------+---------------+
|        2023-01|             10|
|        2023-02|             10|
|        2023-03|              4|
|        2023-04|              7|
|        2023-05|              6|
|        2023-06|              3|
|        2023-07|              3|
|        2023-08|              1|
|        2023-09|              2|
|        2023-10|              2|
+---------------+---------------+



In [17]:
spark.stop()